In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
data = pd.read_csv("data/additional_features.csv")
data

,id,Rank,Title,Primary Artists,Featuring Artists,Date,Danceability,Energy,Loudness,Speechiness,...,lyrics,detected_languages,num_languages,language_confidences,artist_id,explicit,track_popularity,artist_followers,artist_genres,artist_popularity
0,47EiUVwUp4C9fGccaPuUCS,1,dákiti,"bad bunny, jhayco",NaN,26/11/2020,0.731,0.573,-10059.0,0.054,...,"Baby, ya yo me enteré, se nota cuando me ve'\n...",SPANISH,1,SPANISH: 1.0,4q3ewBCX7sLwd24euuV69X,True,78,102185553.0,"reggaeton, trap latino, urbano latino, latin",95.0
1,7igeByaBM0MgGsgXtNxDJ7,1,positions,ariana grande,NaN,29/10/2020,0.736,0.802,-4759.0,0.086,...,Heaven sent you to me\nI'm just hopin’ I don't...,ENGLISH,1,ENGLISH: 1.0,66CXWjxzNUsdJxJ2JdwvnR,True,0,107261237.0,pop,90.0
2,3qQbCzHBycnDpGskqOWY0E,1,ella baila sola,"eslabon armado, peso pluma",NaN,29/05/2023,0.668,0.758,-5176.0,0.033,...,"Compa, ¿qué le parece esa morra?\nLa que anda ...",SPANISH,1,SPANISH: 1.0,0XeEobZplHxzM9QzFQWLiR,False,73,6434994.0,"corridos tumbados, sad sierreño, corrido, sier...",75.0
3,0Yde1VrWbGF2Tus2AQhMuT,1,circles,post malone,NaN,06/09/2019,0.703,0.756,-3407.0,0.038,...,"Oh, oh, oh\nOh, oh, oh\nOh, oh, oh, oh, oh\n...",ENGLISH,1,ENGLISH: 1.0,246dkjvS1zLTtiykXe5h60,False,0,47686394.0,NaN,85.0
4,0TK2YIli7K1leLovkQiNik,1,señorita,"camila cabello, shawn mendes",NaN,25/07/2019,0.759,0.540,-6039.0,0.029,...,I love it when you call me señorita\nI wish I ...,ENGLISH,1,ENGLISH: 1.0,7n2wHs1TKAczGzO7Dd2rGr,False,79,45665812.0,NaN,81.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9156,5FOCkGt7bp22nBVfQ4WSIT,200,indecente,anitta,NaN,19/04/2018,0.874,0.602,-5118.0,0.127,...,"Una mirada te delata, un perfume que te atrapa...",SPANISH,1,SPANISH: 1.0,7FNnA9vBm6EKceENgCGRMb,False,48,13245754.0,"brazilian pop, funk pop, brazilian funk, funk ...",78.0
9157,7zQirOExB0VR8yWUOqYeio,200,dolla sign slime (feat. megan thee stallion),lil nas x,megan thee stallion,27/09/2021,0.923,0.533,-6016.0,0.262,...,Day-Day-Daytrip took it to ten (\nHey\n)\n\nY'...,ENGLISH,1,ENGLISH: 1.0,7jVv8c5Fj3E9VhNjxT4snq,True,50,12311640.0,NaN,71.0
9158,5EgYDyMDCi8X0zDydTF7Rb,200,heartbeat (bts world origil soundtrack),bts,NaN,03/07/2019,0.437,0.716,-4306.0,0.050,...,NaN,NaN,0,NaN,3Nrfpe0tUJi4K4DXYWgMUX,False,0,81105517.0,k-pop,84.0
9159,01vv2AjxgP4uUyb8waYO5Y,200,morph,twenty one pilots,NaN,16/10/2018,0.734,0.607,-7249.0,0.081,...,Can't stop thinking about if and when I die\nF...,ENGLISH,1,ENGLISH: 1.0,3YQKmKGau1PzlVlkL1iodx,False,65,25743995.0,NaN,83.0


In [3]:
metrics = ["Danceability", "Energy", "Loudness", "Speechiness", "Acousticness", "Instrumentalness", "Valence"]

# Prepare dataframe to accumulate sums, sums of squares, and per-metric counts + total Count
cols = ["Artist", "Count"] + [f"Sum {m}" for m in metrics] + [f"SumSq {m}" for m in metrics] + [f"Count {m}" for m in metrics]
artists = pd.DataFrame(columns=cols)

# Helper dict for fast lookup of row index by artist name
artist_idx = {}

# Iterate rows and accumulate
for idx, line in data.iterrows():
    # Collect artists for song
    primary = line["Primary Artists"]
    featuring = line["Featuring Artists"]
    primary_list = [a.strip() for a in str(primary).split(",") if pd.notna(primary) and str(primary).strip()] if pd.notna(primary) else []
    featuring_list = [a.strip() for a in str(featuring).split(",") if pd.notna(featuring) and str(featuring).strip()] if pd.notna(featuring) else []

    for artist in primary_list + featuring_list:
        # If the artist is already in the index we can look it up and then update their entry
        if artist in artist_idx:
            i = artist_idx[artist]
            artists.at[i, "Count"] = int(artists.at[i, "Count"]) + 1
        else:
            i = len(artists)
            artist_idx[artist] = i
            # initialize sums, sumsqs and per-metric counts to zeros
            artists.loc[i] = [artist, 1] + [0.0] * (len(metrics) * 2) + [0] * len(metrics)

        # update per-metric accumulators (only when value is present)
        for m in metrics:
            val = line[m]
            if pd.isna(val):
                continue
            artists.at[i, f"Sum {m}"] = float(artists.at[i, f"Sum {m}"]) + float(val)
            artists.at[i, f"SumSq {m}"] = float(artists.at[i, f"SumSq {m}"]) + float(val) ** 2
            artists.at[i, f"Count {m}"] = int(artists.at[i, f"Count {m}"]) + 1

# Compute mean and std per metric
for m in metrics:
    s = artists[f"Sum {m}"].astype(float)
    ss = artists[f"SumSq {m}"].astype(float)
    cm = artists[f"Count {m}"].astype(int)

    mean = np.where(cm > 0, s / cm, np.nan)
    var = np.where(cm > 1, (ss - (s * s) / cm) / (cm - 1), 0)
    std = np.sqrt(np.maximum(var, 0.0))

    artists[f"Mean {m}"] = mean
    artists[f"Std {m}"] = std

mean_cols = [f"Mean {m}" for m in metrics]
std_cols = [f"Std {m}" for m in metrics]
# Interleave Mean and Std for each metric: Artist, Mean Danceability, Std Danceability, Mean Energy, Std Energy, ...
final_cols = ["Artist"] + [x for m in metrics for x in (f"Mean {m}", f"Std {m}")] + ["Count"]

artists = artists[final_cols].sort_values("Count", ascending=False).reset_index(drop=True)

In [4]:
artists

,Artist,Mean Danceability,Std Danceability,Mean Energy,Std Energy,Mean Loudness,Std Loudness,Mean Speechiness,Std Speechiness,Mean Acousticness,Std Acousticness,Mean Instrumentalness,Std Instrumentalness,Mean Valence,Std Valence,Count
0,drake,0.711716,0.151289,0.549638,0.153531,-6402.349258,3444.709399,0.202100,0.140503,0.200074,0.232417,0.010201,0.065975,0.373162,0.208277,229
1,taylor swift,0.592329,0.113179,0.554347,0.175523,-7408.700833,3317.588195,0.064769,0.058960,0.337579,0.326075,0.004301,0.037129,0.392620,0.189788,216
2,future,0.760352,0.123411,0.582162,0.108463,-5987.031508,2562.616316,0.220631,0.119358,0.104190,0.134780,0.004581,0.030612,0.396207,0.210416,179
3,juice wrld,0.667318,0.108455,0.606345,0.123862,-5771.773986,3311.928867,0.155899,0.138577,0.193385,0.223225,0.002926,0.024425,0.416068,0.191834,148
4,bts,0.639693,0.109827,0.725536,0.144852,-4873.853071,2283.009199,0.108071,0.113135,0.117629,0.162112,0.026800,0.142707,0.513664,0.188043,140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2518,powfu,0.726000,0.000000,0.431000,0.000000,-8765.000000,0.000000,0.135000,0.000000,0.731000,0.000000,0.000000,0.000000,0.348000,0.000000,1
2519,king princess,0.600000,0.000000,0.535000,0.000000,-7043.000000,0.000000,0.107000,0.000000,0.648000,0.000000,0.000000,0.000000,0.269000,0.000000,1
2520,public,0.606000,0.000000,0.799000,0.000000,-4452.000000,0.000000,0.026000,0.000000,0.013000,0.000000,0.000000,0.000000,0.638000,0.000000,1
2521,dj lv mdp,0.628000,0.000000,0.318000,0.000000,-8887.000000,0.000000,0.051000,0.000000,0.807000,0.000000,0.101000,0.000000,0.492000,0.000000,1


In [5]:
def recommend_artists(song_metrics, artists_df=artists, metrics=metrics, min_count=5, top_k=10, eps=1e-6):
    # Prepare song vector
    try:
        song_vec = np.array([float(song_metrics[m]) for m in metrics], dtype=float)
    except Exception:
        raise ValueError("song_metrics must contain all metrics: " + ", ".join(metrics))

    # Candidate artists (filter low-count artists)
    cand = artists_df[artists_df["Count"] >= min_count].copy()
    if cand.empty:
        raise ValueError("No artists with Count >= min_count")

    mean_cols = [f"Mean {m}" for m in metrics]
    std_cols = [f"Std {m}" for m in metrics]

    means = cand[mean_cols].to_numpy(dtype=float) # shape (n_artists, n_metrics)
    stds = cand[std_cols].to_numpy(dtype=float)   # shape (n_artists, n_metrics)
    vars_ = np.maximum(stds ** 2, eps)

    # Assume independent Gaussian per metric with artist mean/std -> compute log-likelihood (higher = better)
    dif = song_vec - means
    # log pdf per metric: -0.5*log(2*pi*var) - 0.5*(dif**2)/var
    log_terms = -0.5 * (np.log(2 * np.pi * vars_) + (dif ** 2) / vars_)
    loglike = np.sum(log_terms, axis=1)
    result = cand.assign(Score=loglike).sort_values("Score", ascending=False)

    # Return top_k with relevant columns
    return result[["Artist", "Count"] + mean_cols + std_cols + ["Score"]].head(top_k)


# Sample 10 songs and print real artists + top 10 guesses
seed = np.random.randint(1000)
mask = data['Primary Artists'].fillna('').str.contains(r'\b(?:billie eilish)\b', case=False, regex=True)
sample = data[mask].copy()

# print(f"Seed = {seed}")
print("-" * 80)

for i, row in sample.iterrows():
    # Build real artist list (primary + featuring)
    primary = row["Primary Artists"]
    featuring = row["Featuring Artists"]
    primary_list = [a.strip() for a in str(primary).split(",") if pd.notna(primary) and str(primary).strip()] if pd.notna(primary) else []
    featuring_list = [a.strip() for a in str(featuring).split(",") if pd.notna(featuring) and str(featuring).strip()] if pd.notna(featuring) else []
    real_artists = primary_list + featuring_list

    # Prepare song metrics and get recommendations
    song_input = {m: row[m] for m in metrics}
    recs = recommend_artists(song_input, artists_df=artists, metrics=metrics, min_count=1, top_k=10)

    # Print results
    print(f"Song: {row['Title']}")
    print("Real artist(s):", ", ".join(real_artists))
    if featuring_list:
        print("Featuring artist(s):", ", ".join(featuring_list))
    print("Top 10 guesses:")
    primary_set = {p.lower().strip() for p in primary_list}
    GREEN = '\033[92m'
    RESET = '\033[0m'
    for rank, (_, r) in enumerate(recs.iterrows(), start=1):
        name = str(r['Artist'])
        if name.lower().strip() in primary_set:
            print(f"  {rank}. {GREEN}{name} (Count={r['Count']}, Score={r['Score']:.4f}){RESET}")
        else:
            print(f"  {rank}. {name} (Count={r['Count']}, Score={r['Score']:.4f})")
    print("-" * 80)

--------------------------------------------------------------------------------
Song: bury a friend
Real artist(s): billie eilish
Top 10 guesses:
  1. billie eilish (Count=58, Score=-8.3282)
  2. xxxtentacion (Count=75, Score=-9.1061)
  3. frank ocean (Count=16, Score=-9.3710)
  4. jackboys (Count=5, Score=-9.7786)
  5. mac miller (Count=31, Score=-11.2504)
  6. metro boomin (Count=82, Score=-12.0636)
  7. damso (Count=31, Score=-13.2085)
  8. beyoncé (Count=58, Score=-13.2459)
  9. drake (Count=229, Score=-13.3700)
  10. offset (Count=43, Score=-13.5319)
--------------------------------------------------------------------------------
Song: wish you were gay
Real artist(s): billie eilish
Top 10 guesses:
  1. dark polo gang (Count=2, Score=0.4362)
  2. bass santa (Count=3, Score=0.1477)
  3. nekfeu (Count=9, Score=0.0538)
  4. rod wave (Count=7, Score=-0.0787)
  5. ynw melly (Count=8, Score=-0.4463)
  6. lil xan (Count=3, Score=-1.0724)
  7. kin$oul (Count=3, Score=-1.1634)
  8. 6lack 

In [6]:
artists.to_csv("data/artists.csv", index=False, encoding="utf-8")